# 전력량 예측 및 MPC 실험 결과 리포트

> `version3_260714_processed`의 저장 결과를 직접 읽어, **예측 모델 성능**과 **MPC 오프라인 평가 결과**를 한 번에 보고합니다.  
> 모든 표와 결론은 노트북 실행 시 원본 JSON/NumPy 결과로부터 다시 생성됩니다.

**핵심 질문**

- 방 1, 방 2, 거실에서 환기·냉난방 전력량을 가장 정확하게 예측한 모델은 무엇인가?
- 저장된 평가 지표가 원본 예측 배열을 재계산한 값과 일치하는가?
- MPC가 오프라인 시뮬레이션에서 에너지를 절감했는가? 해당 결과를 실제 절감 성과로 해석해도 되는가?

In [ ]:
from pathlib import Path
import html
import json

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display


def find_repo_root(start: Path) -> Path:
    # 현재 위치에서 saved_model과 src를 모두 가진 저장소 루트를 찾습니다.
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "saved_model").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "저장소 루트를 찾지 못했습니다. snu-ai-control 저장소 내부에서 실행해 주세요."
    )


REPO_ROOT = find_repo_root(Path.cwd())
DATA_VERSION = "version3_260714_processed"
RUN_VERSION = "version0"
RESULT_ROOT = REPO_ROOT / "saved_model" / DATA_VERSION

ROOM_LABELS = {
    "room1": "방 1",
    "room2": "방 2",
    "livingroom": "거실",
}
MODEL_ORDER = [
    "DLinear", "iTransformer", "MICN", "PatchTST", "TimesNet", "TimeXer"
]
TARGET_LABELS = {
    "forecast_60min_ventilation": "환기유효전력(W)",
    "forecast_60min_hvac": "냉난방유효전력(W)",
}

REPORT_CSS = r'''
<style>
:root {
  --navy: #17365d;
  --blue-100: #eaf3fb;
  --blue-200: #cfe2f3;
  --blue-300: #b8d3ea;
  --red: #e53935;
  --sky: #039be5;
  --green: #157347;
  --amber: #9a6700;
  --gray: #5f6b76;
}
.report-wrap, .report-grid, .report-card, .kpi-grid, .callout, .legend {
  font-family: "Pretendard", "Noto Sans KR", "NanumGothic", Arial, sans-serif;
}
.report-grid {
  display: grid;
  grid-template-columns: repeat(3, minmax(280px, 1fr));
  gap: 24px 28px;
  margin: 14px 0 20px;
}
.report-card { border: 1.5px solid #333; background: white; }
.report-card-title {
  text-align: center; font-size: 15px; font-weight: 800;
  color: #111; padding: 5px 8px; border-bottom: 1.5px solid #333;
}
.result-table, .summary-table {
  width: 100%; border-collapse: collapse; table-layout: fixed;
  font-size: 14px; color: #111;
}
.result-table th, .result-table td,
.summary-table th, .summary-table td {
  border: 1px solid #555; padding: 4px 7px; text-align: center;
  line-height: 1.22;
}
.result-table th, .summary-table th { background: var(--blue-200); font-weight: 800; }
.result-table td.model { background: var(--blue-300); font-weight: 650; }
.result-table td.best { color: var(--red); font-weight: 850; }
.result-table td.second { color: var(--sky); font-weight: 850; }
.legend { font-size: 13px; color: var(--gray); margin: 4px 0 18px; }
.legend .best { color: var(--red); font-weight: 800; }
.legend .second { color: var(--sky); font-weight: 800; }
.callout {
  border-left: 5px solid var(--navy); background: #f5f8fc;
  padding: 13px 16px; margin: 12px 0 18px; line-height: 1.55;
}
.callout.good { border-left-color: var(--green); background: #eef8f2; }
.callout.warn { border-left-color: #d29922; background: #fff8e8; }
.callout.bad { border-left-color: #c62828; background: #fff0f0; }
.kpi-grid {
  display: grid; grid-template-columns: repeat(4, minmax(160px, 1fr));
  gap: 12px; margin: 12px 0 18px;
}
.kpi-card { border: 1px solid #d4dbe3; border-radius: 8px; padding: 13px; background: white; }
.kpi-label { color: var(--gray); font-size: 12px; font-weight: 700; }
.kpi-value { color: var(--navy); font-size: 23px; font-weight: 850; margin-top: 5px; }
.kpi-note { color: var(--gray); font-size: 11px; margin-top: 4px; }
.status-pass { color: var(--green); font-weight: 850; }
.status-fail { color: #c62828; font-weight: 850; }
.status-warn { color: var(--amber); font-weight: 850; }
@media (max-width: 1050px) {
  .report-grid { grid-template-columns: 1fr; }
  .kpi-grid { grid-template-columns: repeat(2, 1fr); }
}
</style>
'''
display(HTML(REPORT_CSS))

print(f"Repository : {REPO_ROOT}")
print(f"Result root: {RESULT_ROOT}")

## 1. 실험 범위와 평가 기준

- **대상 공간:** 방 1, 방 2, 거실
- **예측 대상:** 환기유효전력(W), 냉난방유효전력(W)
- **비교 모델:** DLinear, iTransformer, MICN, PatchTST, TimesNet, TimeXer
- **예측 설정:** 과거 120분 입력으로 이후 60분을 예측 (`seq_len=120`, `pred_len=60`)
- **데이터 분할:** 시간 순서 기반 학습/검증/테스트 = 8:1:1
- **평가 단위:** 저장된 테스트 결과 전체. 중복 슬라이딩 윈도우의 각 예측값을 모두 포함한 집계 지표

전력량 예측 정확도는 다음과 같이 정의합니다.

$$
\text{전력량 정확도}=1-\frac{\text{MAE}}{\operatorname{mean}(E_{actual})},
\qquad
\text{MAE}=\frac{1}{n}\sum_{t=1}^{n}|E_{pred,t}-E_{actual,t}|
$$

$$
\text{MSE}=\frac{1}{n}\sum_{t=1}^{n}(E_{pred,t}-E_{actual,t})^2
$$

정확도는 **높을수록**, MSE는 **낮을수록** 좋습니다. 정확도가 음수라면 MAE가 실제 평균 전력보다 크다는 의미이며 계산 오류나 음의 정확도 퍼센트를 뜻하는 것은 아닙니다.

In [ ]:
def load_forecast_results() -> pd.DataFrame:
    rows = []
    missing = []
    for room_key, room_name in ROOM_LABELS.items():
        for task_key, target_name in TARGET_LABELS.items():
            for model in MODEL_ORDER:
                result_path = (
                    RESULT_ROOT / room_key / model / task_key /
                    RUN_VERSION / "test_results.json"
                )
                if not result_path.is_file():
                    missing.append(result_path)
                    continue
                with result_path.open(encoding="utf-8") as f:
                    result = json.load(f)
                rows.append({
                    "room_key": room_key,
                    "공간": room_name,
                    "task_key": task_key,
                    "예측 대상": target_name,
                    "모델": model,
                    "전력량 정확도": float(result["prediction_accuracy"]),
                    "전력량 정확도(%)": float(result["prediction_accuracy*100"]),
                    "MSE": float(result["mse"]),
                    "MAE": float(result["mae"]),
                    "RMSE": float(result["rmse"]),
                    "result_path": result_path,
                })

    expected = len(ROOM_LABELS) * len(TARGET_LABELS) * len(MODEL_ORDER)
    if missing:
        preview = "\n".join(str(p) for p in missing[:10])
        raise FileNotFoundError(f"누락된 결과가 {len(missing)}개 있습니다:\n{preview}")
    if len(rows) != expected:
        raise AssertionError(f"예상 {expected}개 결과 중 {len(rows)}개만 로드했습니다.")

    frame = pd.DataFrame(rows)
    metric_cols = ["전력량 정확도", "전력량 정확도(%)", "MSE", "MAE", "RMSE"]
    if not np.isfinite(frame[metric_cols].to_numpy()).all():
        raise ValueError("평가 지표에 NaN 또는 무한대가 포함되어 있습니다.")
    return frame


results = load_forecast_results()
expected_count = len(ROOM_LABELS) * len(TARGET_LABELS) * len(MODEL_ORDER)

coverage = (
    results.groupby(["공간", "예측 대상"], sort=False)["모델"]
    .nunique()
    .rename("확인된 모델 수")
    .reset_index()
)
display(HTML(
    f'<div class="callout good"><b>데이터 완전성 확인:</b> '
    f'{len(results)}/{expected_count}개 실험 결과를 로드했습니다. '
    f'각 공간·타겟 조합에 {len(MODEL_ORDER)}개 모델이 모두 존재합니다.</div>'
))
display(coverage)

## 2. 예측 모델 성능 결과

In [ ]:
def leader_row(group: pd.DataFrame) -> dict:
    best_acc = group.loc[group["전력량 정확도"].idxmax()]
    best_mse = group.loc[group["MSE"].idxmin()]
    return {
        "공간": group["공간"].iloc[0],
        "예측 대상": group["예측 대상"].iloc[0],
        "정확도 1위": best_acc["모델"],
        "최고 정확도": best_acc["전력량 정확도"],
        "MSE 1위": best_mse["모델"],
        "최저 MSE": best_mse["MSE"],
    }


leaders = pd.DataFrame([
    leader_row(group)
    for _, group in results.groupby(["room_key", "task_key"], sort=False)
])

same_winner_count = int((leaders["정확도 1위"] == leaders["MSE 1위"]).sum())
display(HTML(
    '<div class="callout"><b>요약:</b> '
    f'총 6개 공간·타겟 조합 중 {same_winner_count}개 조합에서 '
    '정확도와 MSE의 1위 모델이 같습니다. '
    '두 지표가 다른 경우에는 보고 목적에 맞게 정확도와 오차 크기를 함께 확인해야 합니다.</div>'
))

leaders_display = leaders.copy()
leaders_display["최고 정확도"] = leaders_display["최고 정확도"].map(lambda x: f"{x:.4f}")
leaders_display["최저 MSE"] = leaders_display["최저 MSE"].map(lambda x: f"{x:.4f}")
display(leaders_display.reset_index(drop=True))

In [ ]:
def rank_class(value: float, values, higher_is_better: bool) -> str:
    ordered = sorted(set(float(v) for v in values), reverse=higher_is_better)
    if value == ordered[0]:
        return "best"
    if len(ordered) > 1 and value == ordered[1]:
        return "second"
    return ""


def result_table_html(room_key: str, task_key: str) -> str:
    subset = (
        results[(results["room_key"] == room_key) & (results["task_key"] == task_key)]
        .set_index("모델")
        .loc[MODEL_ORDER]
        .reset_index()
    )
    acc_values = subset["전력량 정확도"].tolist()
    mse_values = subset["MSE"].tolist()
    rows_html = []
    for _, row in subset.iterrows():
        acc_class = rank_class(row["전력량 정확도"], acc_values, True)
        mse_class = rank_class(row["MSE"], mse_values, False)
        rows_html.append(
            "<tr>"
            f'<td class="model">{html.escape(str(row["모델"]))}</td>'
            f'<td class="{acc_class}">{row["전력량 정확도"]:.4f}</td>'
            f'<td class="{mse_class}">{row["MSE"]:.4f}</td>'
            "</tr>"
        )
    title = f'{ROOM_LABELS[room_key]} ({TARGET_LABELS[task_key]})'
    return (
        '<section class="report-card">'
        f'<div class="report-card-title">{html.escape(title)}</div>'
        '<table class="result-table">'
        '<thead><tr><th>모델</th><th>전력량 정확도</th><th>MSE</th></tr></thead>'
        f'<tbody>{"".join(rows_html)}</tbody></table></section>'
    )


cards = []
# 예시 레이아웃과 동일하게 상단은 환기, 하단은 냉난방으로 배치합니다.
for task_key in ["forecast_60min_ventilation", "forecast_60min_hvac"]:
    for room_key in ["room1", "room2", "livingroom"]:
        cards.append(result_table_html(room_key, task_key))

display(HTML(
    '<div class="legend">'
    '<span class="best">빨간색: 지표 1위</span> &nbsp;|&nbsp; '
    '<span class="second">파란색: 지표 2위</span>'
    '</div>'
    f'<div class="report-grid">{"".join(cards)}</div>'
))

### 모델별 평균 성능

아래 표는 세 공간의 성능을 단순 평균한 참고용 요약입니다. 공간별 실제 전력 규모가 다르므로 MSE 평균은 공간 간 절대 비교보다 **동일 타겟 내 모델의 대략적인 경향 확인**에 사용합니다.

In [ ]:
overall = (
    results.groupby(["예측 대상", "모델"], sort=False)
    .agg(
        평균_정확도=("전력량 정확도", "mean"),
        평균_MSE=("MSE", "mean"),
    )
    .reset_index()
)
overall["정확도 순위"] = (
    overall.groupby("예측 대상")["평균_정확도"]
    .rank(method="min", ascending=False)
    .astype(int)
)
overall["MSE 순위"] = (
    overall.groupby("예측 대상")["평균_MSE"]
    .rank(method="min", ascending=True)
    .astype(int)
)
overall = overall.sort_values(["예측 대상", "정확도 순위", "MSE 순위"])
overall_display = overall.copy()
overall_display["평균_정확도"] = overall_display["평균_정확도"].map(lambda x: f"{x:.4f}")
overall_display["평균_MSE"] = overall_display["평균_MSE"].map(lambda x: f"{x:.4f}")
display(overall_display.reset_index(drop=True))

## 3. 결과 파일 재계산 검증

`test_results.json`의 MSE·MAE·전력량 정확도를 `TEST_total_outputs.npy`와 `TEST_total_targets.npy`에서 직접 다시 계산합니다. JSON은 소수점 넷째 자리로 저장되므로 허용 오차는 $10^{-4}$로 설정합니다.

In [ ]:
qa_rows = []
for row in results.itertuples(index=False):
    run_dir = Path(row.result_path).parent
    pred_path = run_dir / "TEST_total_outputs.npy"
    true_path = run_dir / "TEST_total_targets.npy"
    if not pred_path.is_file() or not true_path.is_file():
        raise FileNotFoundError(f"검증 배열이 없습니다: {run_dir}")

    pred = np.load(pred_path)
    true = np.load(true_path)
    if pred.shape != true.shape:
        raise ValueError(f"예측/정답 shape 불일치: {run_dir}, {pred.shape} != {true.shape}")

    finite = bool(np.isfinite(pred).all() and np.isfinite(true).all())
    mse_calc = float(np.mean((pred - true) ** 2))
    mae_calc = float(np.mean(np.abs(pred - true)))
    accuracy_calc = float(1.0 - mae_calc / float(np.mean(true)))
    qa_rows.append({
        "공간": row.공간,
        "예측 대상": getattr(row, "_3"),
        "모델": row.모델,
        "shape": str(pred.shape),
        "유한값": finite,
        "MSE 차이": abs(mse_calc - row.MSE),
        "MAE 차이": abs(mae_calc - row.MAE),
        "정확도 차이": abs(accuracy_calc - getattr(row, "_5")),
    })

qa = pd.DataFrame(qa_rows)
max_deltas = qa[["MSE 차이", "MAE 차이", "정확도 차이"]].max()
tolerance = 1e-4
qa_passed = bool(
    qa["유한값"].all()
    and (max_deltas <= tolerance).all()
    and qa["shape"].nunique() == 1
)

status_class = "good" if qa_passed else "bad"
status_text = "공유 가능(검증 통과)" if qa_passed else "검토 필요(검증 실패)"
display(HTML(
    f'<div class="callout {status_class}"><b>QA 상태: {status_text}</b><br>'
    f'검증 실험 수: {len(qa)}개 / 공통 배열 shape: {qa["shape"].iloc[0]}<br>'
    f'최대 절대 차이 — 정확도 {max_deltas["정확도 차이"]:.8f}, '
    f'MSE {max_deltas["MSE 차이"]:.8f}, MAE {max_deltas["MAE 차이"]:.8f}'
    '</div>'
))
assert qa_passed, "원본 배열 재계산 검증에 실패했습니다."

## 4. MPC 오프라인 평가 결과

이 섹션은 `mpc_evaluation.json`이 존재할 때만 표시합니다. 결과는 독립 평가기를 사용한 **반사실적 오프라인 시뮬레이션**이며, 실제 설비에 최적 제어를 적용해 측정한 결과가 아닙니다.

In [ ]:
mpc_paths = sorted(RESULT_ROOT.glob("*/mpc/*/mpc_evaluation.json"))
if not mpc_paths:
    display(HTML('<div class="callout warn">MPC 평가 결과 파일이 없어 이 섹션을 건너뜁니다.</div>'))
    mpc_data = None
else:
    # 현재 저장된 대표 MPC 결과를 사용합니다. 여러 파일이 있으면 첫 파일과 목록을 함께 표시합니다.
    mpc_path = mpc_paths[0]
    with mpc_path.open(encoding="utf-8") as f:
        mpc_data = json.load(f)
    mpc_room_key = mpc_path.relative_to(RESULT_ROOT).parts[0]
    mpc_room_name = ROOM_LABELS.get(mpc_room_key, mpc_room_key)
    print(f"MPC result: {mpc_path}")
    if len(mpc_paths) > 1:
        print(f"참고: MPC 결과 파일이 총 {len(mpc_paths)}개이며 첫 파일을 표시합니다.")

    assessment = mpc_data["quality_assessment"]
    model_passed = bool(assessment["model_validation"]["model_validation_passed"])
    policy_passed = bool(assessment["simulated_policy_acceptance"]["passed"])
    real_status = assessment["real_equipment_performance_claim"]["status"]

    def status_span(passed: bool) -> str:
        cls = "status-pass" if passed else "status-fail"
        label = "PASS" if passed else "FAIL"
        return f'<span class="{cls}">{label}</span>'

    display(HTML(
        '<div class="callout bad">'
        f'<b>{html.escape(mpc_room_name)} MPC 해석 상태: 탐색적 결과</b><br>'
        f'모델 검증 {status_span(model_passed)} &nbsp;|&nbsp; '
        f'오프라인 정책 승인 {status_span(policy_passed)} &nbsp;|&nbsp; '
        f'실장비 검증 <span class="status-warn">{html.escape(real_status.upper())}</span><br>'
        '따라서 아래 에너지 절감률은 오프라인 시뮬레이션 결과이며 실제 절감 성과로 주장할 수 없습니다.'
        '</div>'
    ))

In [ ]:
if mpc_data is not None:
    comparison = mpc_data["policy_evaluation"]["mpc_vs_primary_baseline"]
    ci = comparison["bootstrap_confidence_interval"]
    solver = mpc_data["policy_evaluation"]["solver"]
    protocol = mpc_data["evaluation_protocol"]

    kpis = [
        ("평가 윈도우", f'{mpc_data["evaluated_windows"]:,}개', "윈도우당 60분 폐루프"),
        ("기록 제어 에너지", f'{mpc_data["baseline_energy_wh"]:,.2f} Wh', "primary baseline = recorded"),
        ("MPC 에너지", f'{mpc_data["optimized_energy_wh"]:,.2f} Wh', "독립 평가기 기반"),
        ("모의 절감률", f'{mpc_data["energy_reduction_percent"]:.2f}%', f'95% CI {ci["lower_percent"]:.2f}–{ci["upper_percent"]:.2f}%'),
        ("최적화 횟수", f'{solver["solve_count"]:,}회', f'optimal {solver["optimal_success_count"]:,}회'),
        ("Fallback", f'{solver["fallback_count"]:,}회', "낮을수록 좋음"),
        ("Deadline miss", f'{solver["deadline_miss_count"]:,}회', f'deadline {solver["deadline_seconds"]:.0f}초'),
        ("평균 solve time", f'{solver["solve_seconds"]["mean"]:.2f}초', f'p95 {solver["solve_seconds"]["p95"]:.2f}초'),
    ]
    cards_html = "".join(
        '<div class="kpi-card">'
        f'<div class="kpi-label">{html.escape(label)}</div>'
        f'<div class="kpi-value">{html.escape(value)}</div>'
        f'<div class="kpi-note">{html.escape(note)}</div>'
        '</div>'
        for label, value, note in kpis
    )
    display(HTML(f'<div class="kpi-grid">{cards_html}</div>'))

In [ ]:
if mpc_data is not None:
    violations = mpc_data["policy_evaluation"]["constraint_violations"]
    constraint_rows = []
    for state, recorded in violations["recorded"].items():
        mpc = violations["mpc"][state]
        constraint_rows.append({
            "상태 변수": state,
            "기록 제어 위반량·분": recorded["violation_magnitude_minutes"],
            "MPC 위반량·분": mpc["violation_magnitude_minutes"],
            "위반량·분 변화": mpc["violation_magnitude_minutes"] - recorded["violation_magnitude_minutes"],
            "기록 제어 최대 위반": recorded["maximum_violation"],
            "MPC 최대 위반": mpc["maximum_violation"],
        })
    constraints = pd.DataFrame(constraint_rows)
    numeric_cols = constraints.columns.drop("상태 변수")
    constraints[numeric_cols] = constraints[numeric_cols].round(3)
    display(Markdown("### 제약 위반 비교: 기록 제어 vs MPC"))
    display(constraints)

In [ ]:
if mpc_data is not None:
    assessment = mpc_data["quality_assessment"]
    failed_gates = [
        name
        for name, gate in assessment["model_validation"]["gates"].items()
        if not gate["passed"]
    ]
    support = mpc_data["control_support"]
    non_saving = mpc_data["policy_evaluation"]["mpc_vs_primary_baseline"][
        "window_reduction_percent_distribution"
    ]["non_saving_window_count"]
    display(HTML(
        '<div class="callout warn"><b>MPC 결과 해석</b><ul>'
        f'<li>독립 평가기 기준 기록 제어 대비 에너지는 '
        f'{mpc_data["energy_reduction_percent"]:.2f}% 감소했고, '
        f'비절감 윈도우는 {non_saving}개입니다.</li>'
        f'<li>모델 검증 실패 게이트는 {len(failed_gates)}개입니다: '
        f'{html.escape(", ".join(failed_gates))}</li>'
        f'<li>논리적으로 유효한 제어 조합 {support["logically_valid_combination_count"]}개 중 '
        f'학습 데이터에서 관측된 조합은 {support["training_observed_combination_count"]}개이며, '
        f'미관측 {support["logically_valid_but_unobserved_count"]}개는 최적화에서 제외했습니다.</li>'
        '<li>실장비 적용 전에는 모델 검증 게이트를 먼저 통과시키고, 안전 제약을 포함한 단계적 현장 검증이 필요합니다.</li>'
        '</ul></div>'
    ))

## 5. 보고 시 권장 문구

- **예측 실험:** “세 공간과 두 전력 타겟에서 6개 시계열 모델을 비교했으며, 표의 빨간색은 지표별 1위, 파란색은 2위를 나타낸다. 평가지표는 저장된 원본 예측 배열로 재계산하여 검증했다.”
- **MPC 실험:** “독립 평가기를 사용한 오프라인 시뮬레이션에서는 기록 제어 대비 에너지 감소 가능성이 관찰되었다. 다만 모델 품질 게이트를 통과하지 못했고 실제 설비 적용 전이므로, 현재 결과는 탐색적 근거로만 해석한다.”

### 재실행 방법

저장소의 `src/jupyter` 디렉터리에서 이 노트북을 열고 **Run All**을 실행합니다. 같은 경로 구조로 결과 JSON과 NumPy 배열이 갱신되면 표, 순위, QA 결과가 자동으로 갱신됩니다.